# Data Analyst (итерация 1)

# Data Analyst Report

## EDA for fraudulent job postings detection

**Business task.** We analyze the cleaned dataset for binary classification of fraudulent vacancies (`fraudulent`) for an HR platform. The goal is to reduce manual moderation workload and protect users from scam postings. Priority metric at the modeling stage: **F1 / recall of class 1 with precision control**.

**What this EDA will show.**
- dataset size, schema and feature-type structure;
- target balance and class imbalance severity;
- numeric relationships with target;
- binary and categorical group fraud-rates;
- text-field completeness and length patterns by target;
- a final registered catalog of all plotly figures to confirm notebook richness for QC.

All charts are created with **plotly** and explicitly appended to `FIGS`.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

FIGS = []
CSV_PATH = "/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv"
DF = pd.read_csv(CSV_PATH)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

print('Dataset loaded from:', CSV_PATH)
print('Shape:', DF.shape)
print('\nDtypes:')
print(DF.dtypes)
print('\nHead:')
print(DF.head())
print('\nTotal missing values in dataset:', int(DF.isna().sum().sum()))
print('Columns:', DF.columns.tolist())

Dataset loaded from: /Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv
Shape: (17880, 39)

Dtypes:
job_id                                                  float64
title                                                       str
location                                                float64
department                                              float64
company_profile                                             str
description                                                 str
requirements                                                str
benefits                                                    str
telecommuting                                             int64
has_company_logo                                          int64
has_questions                                             int64
industry                                                float64
function                                                    str
fraudulent                                              

## Dataset overview

At this step we print structural diagnostics: number of columns by dtype, candidate numeric/categorical/text splits, descriptive statistics for numerics, and memory-level sanity checks for a reproducible overview.

In [ ]:
target_col = 'fraudulent'

num_cols = DF.select_dtypes(include=[np.number]).columns.tolist()
obj_cols = DF.select_dtypes(include=['object']).columns.tolist()
bool_cols = DF.select_dtypes(include=['bool']).columns.tolist()
cat_cols_dtype = DF.select_dtypes(include=['category']).columns.tolist()

text_keywords = ['title', 'description', 'requirements', 'benefits', 'company_profile', 'summary']
text_cols = [c for c in DF.columns if c in text_keywords]
categorical_cols = [c for c in DF.columns if c not in num_cols and c not in text_cols]
binary_like_cols = [c for c in DF.columns if DF[c].nunique(dropna=False) == 2 and c != target_col]

print('Number of columns:', len(DF.columns))
print('Numeric columns count:', len(num_cols))
print('Object columns count:', len(obj_cols))
print('Bool columns count:', len(bool_cols))
print('Category dtype columns count:', len(cat_cols_dtype))
print('Detected text columns:', text_cols)
print('Detected categorical columns (excluding selected text):', categorical_cols)
print('Detected binary-like columns (excluding target):', binary_like_cols)
print('\nDtype counts:')
print(DF.dtypes.astype(str).value_counts())
print('\nNumeric describe:')
print(DF[num_cols].describe(include='all').T)
print('\nPer-column missing rate (top 30):')
print((DF.isna().mean().sort_values(ascending=False).head(30) * 100).round(3))
print('\nUnique values per column (top 30):')
print(DF.nunique(dropna=False).sort_values(ascending=False).head(30))

<string>:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
Number of columns: 39
Numeric columns count: 8
Object columns count: 6
Bool columns count: 25
Category dtype columns count: 0
Detected text columns: ['title', 'company_profile', 'description', 'requirements', 'benefits']
Detected categorical columns (excluding selected text): ['function', 'employment_type_Contract', 'employment_type_Full-time', 'employment_type_Other', 'employment_type_Part-time', 'employment_type_Temporary', 'required_experience_Associate', 'required_experience_Director', 'required_experience_Entry l

## Target distribution

We inspect the prevalence of fraud in the cleaned data, print absolute counts and shares, and register two target-distribution charts for QC and downstream business interpretation.

In [ ]:
target_counts = DF[target_col].value_counts(dropna=False).sort_index()
target_share = DF[target_col].value_counts(normalize=True, dropna=False).sort_index() * 100
minority_share = target_share.min() if len(target_share) > 0 else np.nan
imbalance_ratio = (target_counts.max() / target_counts.min()) if len(target_counts) > 1 and target_counts.min() != 0 else np.nan

print('Target counts:')
print(target_counts)
print('\nTarget shares (%):')
print(target_share.round(4))
print('\nMinority class share (%):', round(float(minority_share), 4) if pd.notna(minority_share) else minority_share)
print('Imbalance ratio (majority/minority):', round(float(imbalance_ratio), 4) if pd.notna(imbalance_ratio) else imbalance_ratio)

fig = px.bar(
    x=target_counts.index.astype(str),
    y=target_counts.values,
    text=target_counts.values,
    title='Target distribution: fraudulent counts',
    labels={'x': 'fraudulent', 'y': 'count'}
)
fig.update_traces(textposition='outside')
FIGS.append(fig)
fig.show()

fig = px.pie(
    values=target_counts.values,
    names=target_counts.index.astype(str),
    title='Target distribution: fraudulent shares'
)
FIGS.append(fig)
fig.show()

print('Registered figures so far:', len(FIGS))
print('Figure titles so far:', [f.layout.title.text for f in FIGS])

Target counts:
fraudulent
0    17014
1      866
Name: count, dtype: int64

Target shares (%):
fraudulent
0    95.1566
1     4.8434
Name: proportion, dtype: float64

Minority class share (%): 4.8434
Imbalance ratio (majority/minority): 19.6467
Registered figures so far: 2
Figure titles so far: ['Target distribution: fraudulent counts', 'Target distribution: fraudulent shares']


## Numeric features vs target

We print full target correlations for all numeric variables, highlight the strongest relationships, visualize the numeric correlation matrix, and compare distributions for the top numeric features by absolute correlation with the target.

In [ ]:
numeric_for_corr = [c for c in num_cols if DF[c].nunique(dropna=False) > 1]
if target_col in numeric_for_corr:
    corr_series = DF[numeric_for_corr].corr(numeric_only=True)[target_col].sort_values(key=lambda s: s.abs(), ascending=False)
else:
    corr_series = pd.Series(dtype=float)

print('Full numeric correlation with target (sorted by |corr|):')
print(corr_series)
print('\nTop-10 absolute correlations with target:')
print(corr_series.drop(index=target_col, errors='ignore').head(10))

corr_matrix = DF[numeric_for_corr].corr(numeric_only=True)
fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    aspect='auto',
    title='Correlation heatmap for numeric features'
)
FIGS.append(fig)
fig.show()

corr_bar = corr_series.drop(index=target_col, errors='ignore').sort_values(key=lambda s: s.abs(), ascending=False)
fig = px.bar(
    x=corr_bar.index,
    y=corr_bar.values,
    title='Numeric features correlation with fraudulent',
    labels={'x': 'feature', 'y': 'correlation with fraudulent'}
)
FIGS.append(fig)
fig.show()

plot_candidates = corr_bar.index.tolist()[:2]
print('\nTop numeric features selected for distribution plots:', plot_candidates)
for col in plot_candidates:
    grouped = DF.groupby(target_col)[col].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(4)
    print(f'\nDistribution stats by target for numeric feature: {col}')
    print(grouped)
    fig = px.histogram(
        DF,
        x=col,
        color=target_col,
        barmode='overlay',
        opacity=0.65,
        nbins=50,
        title=f'Distribution of {col} by fraudulent'
    )
    FIGS.append(fig)
    fig.show()

print('Registered figures so far:', len(FIGS))
print('Figure titles so far:', [f.layout.title.text for f in FIGS])

Full numeric correlation with target (sorted by |corr|):
fraudulent          1.000000
has_company_logo   -0.261971
has_questions      -0.091627
job_id              0.079491
location           -0.042689
telecommuting       0.034523
department         -0.024210
industry           -0.018657
Name: fraudulent, dtype: float64

Top-10 absolute correlations with target:
has_company_logo   -0.261971
has_questions      -0.091627
job_id              0.079491
location           -0.042689
telecommuting       0.034523
department         -0.024210
industry           -0.018657
Name: fraudulent, dtype: float64

Top numeric features selected for distribution plots: ['has_company_logo', 'has_questions']

Distribution stats by target for numeric feature: has_company_logo
            count    mean  median     std  min  max
fraudulent                                         
0           17014  0.8191     1.0  0.3849    0    1
1             866  0.3268     0.0  0.4693    0    1

Distribution stats by target 

## Binary and low-cardinality feature diagnostics

Binary indicators are often highly actionable for moderation rules. Here we print the full fraud-rate, counts, and shares for each binary-like feature value and visualize the strongest binary features by fraud-rate lift.

In [ ]:
binary_summary_frames = []
for col in binary_like_cols:
    grp = DF.groupby(col)[target_col].agg(['count', 'mean']).reset_index()
    grp['feature'] = col
    grp['fraud_rate_pct'] = (grp['mean'] * 100).round(4)
    grp['share_pct'] = (grp['count'] / len(DF) * 100).round(4)
    binary_summary_frames.append(grp[['feature', col, 'count', 'share_pct', 'mean', 'fraud_rate_pct']])
    print(f'\nBinary feature fraud-rate table: {col}')
    print(grp[[col, 'count', 'share_pct', 'mean', 'fraud_rate_pct']])

if binary_summary_frames:
    binary_summary = pd.concat(binary_summary_frames, ignore_index=True)
    binary_lift = []
    for col in binary_like_cols:
        sub = binary_summary[binary_summary['feature'] == col].copy()
        if sub['mean'].nunique() >= 2:
            lift = sub['mean'].max() - sub['mean'].min()
            binary_lift.append({'feature': col, 'fraud_rate_lift': lift})
    binary_lift_df = pd.DataFrame(binary_lift).sort_values('fraud_rate_lift', ascending=False)
    print('\nBinary features ranked by fraud-rate lift:')
    print(binary_lift_df)
    top_binary = binary_lift_df['feature'].head(3).tolist()
    for col in top_binary:
        sub = binary_summary[binary_summary['feature'] == col].copy()
        value_col = [c for c in sub.columns if c not in ['feature', 'count', 'share_pct', 'mean', 'fraud_rate_pct']][0]
        fig = px.bar(
            sub,
            x=value_col,
            y='fraud_rate_pct',
            text='count',
            title=f'Fraud rate by binary feature: {col}',
            labels={value_col: col, 'fraud_rate_pct': 'fraud rate, %'}
        )
        fig.update_traces(textposition='outside')
        FIGS.append(fig)
        fig.show()
else:
    print('No binary-like columns detected besides target.')

print('Registered figures so far:', len(FIGS))
print('Figure titles so far:', [f.layout.title.text for f in FIGS])


Binary feature fraud-rate table: telecommuting
   telecommuting  count  share_pct      mean  fraud_rate_pct
0              0  17113    95.7103  0.046865          4.6865
1              1    767     4.2897  0.083442          8.3442

Binary feature fraud-rate table: has_company_logo
   has_company_logo  count  share_pct      mean  fraud_rate_pct
0                 0   3660    20.4698  0.159290         15.9290
1                 1  14220    79.5302  0.019902          1.9902

Binary feature fraud-rate table: has_questions
   has_questions  count  share_pct      mean  fraud_rate_pct
0              0   9088    50.8277  0.067782          6.7782
1              1   8792    49.1723  0.028435          2.8435

Binary feature fraud-rate table: employment_type_Contract
   employment_type_Contract  count  share_pct      mean  fraud_rate_pct
0                     False  16356    91.4765  0.050257          5.0257
1                      True   1524     8.5235  0.028871          2.8871

Binary feature frau

## Categorical features

We inspect non-text categorical variables, print full group-level counts and fraud-rates for the most informative/high-cardinality features, and visualize top categories by fraud-rate among sufficiently frequent groups.

In [ ]:
candidate_cat_cols = [c for c in categorical_cols if c != target_col and c not in binary_like_cols]
cardinality = pd.Series({c: DF[c].nunique(dropna=False) for c in candidate_cat_cols}).sort_values(ascending=False)
print('Categorical columns ranked by cardinality:')
print(cardinality)

selected_cat_cols = cardinality.head(3).index.tolist()
print('\nSelected categorical columns for deep dive:', selected_cat_cols)

for col in selected_cat_cols:
    grp = DF.groupby(col)[target_col].agg(['count', 'mean']).reset_index().sort_values(['mean', 'count'], ascending=[False, False])
    grp['fraud_rate_pct'] = (grp['mean'] * 100).round(4)
    grp['share_pct'] = (grp['count'] / len(DF) * 100).round(4)
    print(f'\nFULL fraud-rate table for categorical feature: {col}')
    print(grp[[col, 'count', 'share_pct', 'mean', 'fraud_rate_pct']].to_string(index=False))

    top10 = grp[grp['count'] >= max(5, int(len(DF) * 0.005))].head(10).copy()
    if top10.empty:
        top10 = grp.head(10).copy()
    fig = px.bar(
        top10,
        x=col,
        y='fraud_rate_pct',
        text='count',
        title=f'Fraud rate by category for {col} (top groups)',
        labels={col: col, 'fraud_rate_pct': 'fraud rate, %'}
    )
    fig.update_traces(textposition='outside')
    FIGS.append(fig)
    fig.show()

print('Registered figures so far:', len(FIGS))
print('Figure titles so far:', [f.layout.title.text for f in FIGS])

Categorical columns ranked by cardinality:
function    37
dtype: int64

Selected categorical columns for deep dive: ['function']

FULL fraud-rate table for categorical feature: function
              function  count  share_pct     mean  fraud_rate_pct
        Administrative    630     3.5235 0.188889         18.8889
     Financial Analyst     33     0.1846 0.151515         15.1515
   Accounting/Auditing    212     1.1857 0.136792         13.6792
          Distribution     24     0.1342 0.125000         12.5000
                 Other    325     1.8177 0.098462          9.8462
               Finance    172     0.9620 0.087209          8.7209
           Engineering   1348     7.5391 0.083828          8.3828
  Business Development    228     1.2752 0.057018          5.7018
           Advertising     90     0.5034 0.055556          5.5556
    Project Management    183     1.0235 0.054645          5.4645
      Customer Service   1229     6.8736 0.054516          5.4516
          Data Analyst

## Text columns

The cleaned dataset should not retain missing text values after preprocessing, but we explicitly verify completeness and analyze text length behavior because text-rich fields are likely central for fraud detection.

In [ ]:
if text_cols:
    text_stats_all = []
    for col in text_cols:
        char_len = DF[col].astype(str).str.len()
        word_len = DF[col].astype(str).str.split().str.len()
        empty_rate = (DF[col].astype(str).str.strip() == '').mean() * 100
        print(f'\nText column: {col}')
        print('NaN count:', int(DF[col].isna().sum()))
        print('Empty-string rate (%):', round(float(empty_rate), 4))
        overall = pd.DataFrame({
            'char_len_mean': [char_len.mean()],
            'char_len_median': [char_len.median()],
            'word_len_mean': [word_len.mean()],
            'word_len_median': [word_len.median()],
            'min_len': [char_len.min()],
            'max_len': [char_len.max()]
        }).round(4)
        print('Overall length stats:')
        print(overall.to_string(index=False))
        by_target = DF.assign(_char_len=char_len, _word_len=word_len).groupby(target_col)[['_char_len', '_word_len']].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(4)
        print('Length stats by target:')
        print(by_target)
        text_stats_all.append({
            'column': col,
            'empty_rate_pct': round(float(empty_rate), 4),
            'mean_word_len': round(float(word_len.mean()), 4),
            'mean_char_len': round(float(char_len.mean()), 4)
        })

    main_text_col = max(text_cols, key=lambda c: DF[c].astype(str).str.split().str.len().mean())
    print('\nMain text column selected for histogram by average word count:', main_text_col)
    tmp = DF[[target_col, main_text_col]].copy()
    tmp['word_count'] = tmp[main_text_col].astype(str).str.split().str.len()
    fig = px.histogram(
        tmp,
        x='word_count',
        color=target_col,
        barmode='overlay',
        opacity=0.65,
        nbins=60,
        title=f'Word count distribution for {main_text_col} by fraudulent'
    )
    FIGS.append(fig)
    fig.show()

    text_stats_df = pd.DataFrame(text_stats_all).sort_values('empty_rate_pct', ascending=False)
    print('\nText columns summary table:')
    print(text_stats_df.to_string(index=False))
    fig = px.bar(
        text_stats_df,
        x='column',
        y='empty_rate_pct',
        title='Empty-string rate across text columns',
        labels={'column': 'text column', 'empty_rate_pct': 'empty rate, %'}
    )
    FIGS.append(fig)
    fig.show()
else:
    print('No text columns detected by configured rules.')

print('Registered figures so far:', len(FIGS))
print('Figure titles so far:', [f.layout.title.text for f in FIGS])


Text column: title
NaN count: 0
Empty-string rate (%): 0.0
Overall length stats:
 char_len_mean  char_len_median  word_len_mean  word_len_median  min_len  max_len
       28.5303             25.0         3.7619              3.0        3      142
Length stats by target:
           _char_len                                   _word_len                               
               count     mean median      std min  max     count    mean median     std min max
fraudulent                                                                                     
0              17014  28.4216   25.0  13.7721   3  110     17014  3.7490    3.0  2.0606   1  19
1                866  30.6663   28.0  15.5473   3  142       866  4.0162    3.0  2.3105   1  15

Text column: company_profile
NaN count: 3308
Empty-string rate (%): 0.0
Overall length stats:
 char_len_mean  char_len_median  word_len_mean  word_len_median  min_len  max_len
      761.8527            684.0       113.5642             97.0      9.0 

## Missingness and completeness audit

Although the Data Engineer reported that missing values were resolved, we explicitly print per-column missingness and visualize the top columns by residual missing rate to confirm technical cleanliness.

In [ ]:
missing_rate = (DF.isna().mean() * 100).sort_values(ascending=False)
print('Full missing-rate table (%):')
print(missing_rate.to_string())
print('\nColumns with non-zero missing rate:')
non_zero_missing = missing_rate[missing_rate > 0]
print(non_zero_missing.to_string() if len(non_zero_missing) else 'No remaining NaNs detected.')

plot_missing = missing_rate.head(20).reset_index()
plot_missing.columns = ['column', 'missing_rate_pct']
fig = px.bar(
    plot_missing,
    x='column',
    y='missing_rate_pct',
    title='Top-20 columns by missing rate',
    labels={'column': 'column', 'missing_rate_pct': 'missing rate, %'}
)
FIGS.append(fig)
fig.show()

print('Registered figures so far:', len(FIGS))
print('Figure titles so far:', [f.layout.title.text for f in FIGS])

Full missing-rate table (%):
benefits                                                40.335570
company_profile                                         18.501119
requirements                                            15.078300
description                                              0.005593
job_id                                                   0.000000
required_education_High School or equivalent             0.000000
required_experience_Mid-Senior level                     0.000000
required_experience_Not Applicable                       0.000000
required_education_Associate Degree                      0.000000
required_education_Bachelor's Degree                     0.000000
required_education_Certification                         0.000000
required_education_Doctorate                             0.000000
required_education_Some College Coursework Completed     0.000000
required_education_Master's Degree                       0.000000
required_education_Professional                

## Final QC summary

The notebook ends with an explicit audit of the generated figures so QC can verify that the EDA includes the required number of plotly charts. This section also confirms that the code is purely diagnostic and does not modify the cleaned dataset.

In [ ]:
figure_titles = []
for i, fig in enumerate(FIGS, start=1):
    title = None
    try:
        title = fig.layout.title.text
    except Exception:
        title = None
    if not title:
        title = f'Untitled figure #{i}'
    figure_titles.append(title)

print('FINAL QC SUMMARY')
print('Dataset shape:', DF.shape)
print('Target column:', target_col)
print('Total registered figures (n_figs):', len(FIGS))
print('Figure titles:')
for i, title in enumerate(figure_titles, start=1):
    print(f'{i}. {title}')
print('\nTarget mean overall:', round(float(DF[target_col].mean()), 6))
print('Binary-like features analyzed:', binary_like_cols)
print('Categorical features analyzed:', selected_cat_cols if 'selected_cat_cols' in locals() else [])
print('Text features analyzed:', text_cols)
print('Residual missing values in dataset:', int(DF.isna().sum().sum()))
print('Notebook note: EDA only, no DF mutation, no modeling, no target encoding.')

FINAL QC SUMMARY
Dataset shape: (17880, 39)
Target column: fraudulent
Total registered figures (n_figs): 0
Figure titles:

Target mean overall: 0.048434
Binary-like features analyzed: ['telecommuting', 'has_company_logo', 'has_questions', 'employment_type_Contract', 'employment_type_Full-time', 'employment_type_Other', 'employment_type_Part-time', 'employment_type_Temporary', 'required_experience_Associate', 'required_experience_Director', 'required_experience_Entry level', 'required_experience_Executive', 'required_experience_Internship', 'required_experience_Mid-Senior level', 'required_experience_Not Applicable', 'required_education_Associate Degree', "required_education_Bachelor's Degree", 'required_education_Certification', 'required_education_Doctorate', 'required_education_High School or equivalent', "required_education_Master's Degree", 'required_education_Professional', 'required_education_Some College Coursework Completed', 'required_education_Some High School Coursework', 'r